# 🍌 Banana Leaf Disease Detector — Multi-Label (Image Processing + Classical ML)

**Method:** classical **image processing** (leaf/lesion color segmentation) + **handcrafted features**
(HSV color moments + GLCM texture) + **traditional machine learning** (SVM / Random Forest).
No deep learning / CNNs are used anywhere in this notebook.

**What's new vs. the previous version:**
- Handles all **3** dataset folders: `SINGLE_LABEL` (4 classes), `MULTI_LABEL` (2 diseases on one leaf),
  and `NEGATIVE` (non-leaf photos that must never fire a detection).
- Produces a **multi-label** result — a leaf can be flagged with more than one disease at once.
- Adds an **"Is this a banana leaf?" gate** so random/non-leaf photos are rejected before disease
  scoring ever runs.
- Exports models both as **`.pkl`** (for Python re-use) and **`.onnx`** (for the Android app), plus a
  plain-JSON scaler so the exact same math can run outside Python.

**Pipeline overview**
1. Segment each image into a *leaf region* and a *lesion (diseased-spot) region* using Otsu + HSV +
   K-means color clustering (pure image processing, no learning involved).
2. Extract 33 handcrafted features per image: color moments (mean/std/skew in H,S,V) on both regions,
   GLCM texture (contrast, dissimilarity, homogeneity, energy, correlation, ASM) on both regions, and
   3 shape/coverage ratios (leaf coverage, lesion coverage, edge density).
3. Train an **"IsLeaf" gate classifier** (banana leaf vs. random negative photo).
4. Train **3 independent binary classifiers** (Binary Relevance), one per disease — Cordana, Sigatoka,
   Pestalotiopsis — each trained with SVM and Random Forest via grid search, best one kept.
5. Tune a probability threshold per classifier on the validation set.
6. Evaluate the full multi-label pipeline on a held-out test set.
7. Export everything needed for the Android app.


In [ ]:
!pip install -q scikit-image tqdm joblib skl2onnx onnxruntime

## 1. Get the dataset
Mounts your Drive, and either finds a previously-saved copy of the dataset zip or lets you upload it once (then remembers it for next time).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, shutil

DRIVE_DIR = "/content/drive/MyDrive/bananaleaf_dataset"
DRIVE_ZIP = f"{DRIVE_DIR}/bananaleaf_dataset.zip"
os.makedirs(DRIVE_DIR, exist_ok=True)

if os.path.exists(DRIVE_ZIP):
    print(f"Found dataset already saved in Drive: {DRIVE_ZIP}")
else:
    print("Not found in Drive yet — pick your bananaleaf_dataset(_1_).zip from your computer.")
    from google.colab import files
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    shutil.copy(fname, DRIVE_ZIP)
    print(f"Saved a copy to Drive at {DRIVE_ZIP} — you won't need to re-upload it again.")

if not os.path.exists("/content/bananaleaf_dataset"):
    with zipfile.ZipFile(DRIVE_ZIP, "r") as z:
        z.extractall("/content/bananaleaf_dataset")
    print("Extracted to /content/bananaleaf_dataset")
else:
    print("/content/bananaleaf_dataset already present this session.")

In [ ]:
import glob

def find_dataset_root(root="/content/bananaleaf_dataset"):
    """Locate the folder that directly contains SINGLE_LABEL / MULTI_LABEL / NEGATIVE."""
    need = {"single_label", "multi_label", "negative"}
    for dirpath, dirnames, _ in os.walk(root):
        if need.issubset({d.lower() for d in dirnames}):
            return dirpath
    return None

DATASET_ROOT = find_dataset_root()
print("Detected dataset root:", DATASET_ROOT)
assert DATASET_ROOT is not None, "Could not find SINGLE_LABEL / MULTI_LABEL / NEGATIVE folders — check the zip contents."

def resolve(name):
    matches = [d for d in os.listdir(DATASET_ROOT) if d.lower() == name.lower()]
    return os.path.join(DATASET_ROOT, matches[0])

SINGLE_DIR = resolve("SINGLE_LABEL")
MULTI_DIR = resolve("MULTI_LABEL")
NEGATIVE_DIR = resolve("NEGATIVE")

print("\nSINGLE_LABEL classes:")
for f in sorted(os.listdir(SINGLE_DIR)):
    print(f"  {f:20s} {len(os.listdir(os.path.join(SINGLE_DIR, f)))} images")
print("\nMULTI_LABEL combos:")
for f in sorted(os.listdir(MULTI_DIR)):
    print(f"  {f:35s} {len(os.listdir(os.path.join(MULTI_DIR, f)))} images")
print("\nNEGATIVE subfolders:")
for f in sorted(os.listdir(NEGATIVE_DIR)):
    print(f"  {f:20s} {len(os.listdir(os.path.join(NEGATIVE_DIR, f)))} images")

## 2. Build a unified multi-label index
Every image gets a 4-way multi-hot label `[Healthy, Cordana, Sigatoka, Pestalotiopsis]` plus an `is_leaf` flag. Negative (non-leaf) images get all-zero disease labels **and** `is_leaf = 0`, so the model can learn to reject them outright rather than just calling them "Healthy".

In [ ]:
DISEASE_LABELS = ["Cordana", "Sigatoka", "Pestalotiopsis"]
ALL_LABELS = ["Healthy"] + DISEASE_LABELS

def parse_label_from_folder(folder_name):
    name = folder_name.lower()
    if name in ("background", "objects", "other_plant", "person"):
        return {c: 0 for c in ALL_LABELS}, False
    labels = {c: 0 for c in ALL_LABELS}
    if name == "healthy":
        labels["Healthy"] = 1
        return labels, True
    hit = False
    for d in DISEASE_LABELS:
        if d.lower() in name:
            labels[d] = 1
            hit = True
    if not hit:
        raise ValueError(f"Unrecognized folder: {folder_name}")
    return labels, True

records = []  # (path, folder, is_leaf, Healthy, Cordana, Sigatoka, Pestalotiopsis)
for top_dir in [SINGLE_DIR, MULTI_DIR, NEGATIVE_DIR]:
    for folder in sorted(os.listdir(top_dir)):
        folder_dir = os.path.join(top_dir, folder)
        if not os.path.isdir(folder_dir):
            continue
        labels, is_leaf = parse_label_from_folder(folder)
        for p in glob.glob(os.path.join(folder_dir, "*")):
            records.append([p, folder, int(is_leaf)] + [labels[c] for c in ALL_LABELS])

import pandas as pd
index_df = pd.DataFrame(records, columns=["path", "folder", "is_leaf"] + ALL_LABELS)
print("Total images:", len(index_df))
index_df.groupby("folder").size()

## 3. Preview samples

In [ ]:
import cv2
import matplotlib.pyplot as plt

preview_folders = ["Healthy", "Cordana", "Sigatoka", "Pestalotiopsis",
                    "cordana_sigatoka", "pestalotiopsis_sigatoka", "background", "person"]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, folder in zip(axes.flat, preview_folders):
    row = index_df[index_df["folder"].str.lower() == folder.lower()].iloc[0]
    img = cv2.cvtColor(cv2.imread(row["path"]), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(folder, fontsize=11)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Image processing — leaf & lesion segmentation

Pure classical CV, no learning:
1. **Otsu thresholding** on grayscale to separate the subject from the background.
2. **Leaf mask** = foreground pixels that also fall in a broad green/olive/brown plant-tissue hue band
   in HSV. This is useful on its own signal for rejecting non-leaf photos later.
3. **Lesion mask** = run K-means (k=3) on the foreground pixels' HSV values, then pick the cluster whose
   hue is farthest from "healthy green" — that's the diseased-looking cluster.

In [ ]:
IMG_SIZE = 200
GREEN_HUE = 60      # OpenCV hue scale is 0-179; green sits around 60
KMEANS_K = 3

import numpy as np

def preprocess(img_bgr, size=IMG_SIZE):
    img = cv2.resize(img_bgr, (size, size), interpolation=cv2.INTER_AREA)
    img = cv2.GaussianBlur(img, (5, 5), 0)
    return img


def segment_leaf_and_lesion(img_bgr, k=KMEANS_K):
    """Returns (img, hsv, leaf_mask, lesion_mask)."""
    img = preprocess(img_bgr)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    _, otsu_mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if otsu_mask.mean() > 127:
        otsu_mask = cv2.bitwise_not(otsu_mask)
    if otsu_mask.sum() / 255 < 0.05 * img.size / 3:
        otsu_mask = np.full_like(gray, 255)

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # leaf mask: foreground AND a broad green/olive/brown plant-tissue hue band
    h = hsv[:, :, 0].astype(np.int32)
    s = hsv[:, :, 1]
    v = hsv[:, :, 2]
    plant_hue = (h >= 15) & (h <= 100)
    plant_sv = (s > 25) & (v > 25)
    leaf_mask = ((otsu_mask > 0) & plant_hue & plant_sv).astype(np.uint8) * 255
    leaf_mask = cv2.morphologyEx(leaf_mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    if leaf_mask.sum() / 255 < 20:
        leaf_mask = otsu_mask.copy()

    # lesion mask: k-means on foreground HSV, cluster farthest from green hue
    fg_flat = otsu_mask.reshape(-1) > 0
    Z_all = hsv.reshape((-1, 3)).astype(np.float32)
    Z_fg = Z_all[fg_flat]

    lesion_mask = np.zeros((IMG_SIZE * IMG_SIZE,), dtype=np.uint8)
    if len(Z_fg) >= k:
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 0.5)
        _, labels_fg, centers = cv2.kmeans(Z_fg, k, None, criteria, 5, cv2.KMEANS_PP_CENTERS)
        labels_fg = labels_fg.flatten()
        hue_dist = np.abs(centers[:, 0] - GREEN_HUE)
        lesion_cluster = int(np.argmax(hue_dist))
        lesion_mask[fg_flat] = np.where(labels_fg == lesion_cluster, 255, 0).astype(np.uint8)
    lesion_mask = lesion_mask.reshape(IMG_SIZE, IMG_SIZE)

    lesion_mask = cv2.bitwise_and(lesion_mask, otsu_mask)
    lesion_mask = cv2.morphologyEx(lesion_mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    if lesion_mask.sum() / 255 < 20:
        lesion_mask = np.zeros_like(lesion_mask)

    return img, hsv, leaf_mask, lesion_mask

In [ ]:
# Visual sanity check of the segmentation on a few samples
fig, axes = plt.subplots(3, 4, figsize=(16, 11))
sample_rows = pd.concat([
    index_df[index_df["folder"] == "Cordana"].iloc[[0]],
    index_df[index_df["folder"] == "Sigatoka"].iloc[[0]],
    index_df[index_df["folder"] == "cordana_pestalotiopsis"].iloc[[0]],
])
for row_i, (_, row) in enumerate(sample_rows.iterrows()):
    img_bgr = cv2.imread(row["path"])
    img, hsv, leaf_mask, lesion_mask = segment_leaf_and_lesion(img_bgr)
    axes[row_i, 0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); axes[row_i, 0].set_title(f'{row["folder"]} — original')
    axes[row_i, 1].imshow(leaf_mask, cmap="Greens"); axes[row_i, 1].set_title("leaf mask")
    axes[row_i, 2].imshow(lesion_mask, cmap="Reds"); axes[row_i, 2].set_title("lesion mask")
    overlay = img.copy()
    overlay[lesion_mask > 0] = [0, 0, 255]
    axes[row_i, 3].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)); axes[row_i, 3].set_title("lesion overlay")
    for ax in axes[row_i]:
        ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Feature extraction

33 handcrafted features per image:
- **9** color moments (mean, std, skew × H,S,V) on the **leaf** region
- **9** color moments on the **lesion** region
- **6** GLCM texture features (contrast, dissimilarity, homogeneity, energy, correlation, ASM) on the **leaf** region
- **6** GLCM texture features on the **lesion** region
- **3** shape/coverage ratios: `leaf_ratio`, `lesion_ratio`, `edge_density`

In [ ]:
from skimage.feature import graycomatrix, graycoprops

GLCM_PROPS = ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]

FEATURE_NAMES = (
    [f"Leaf_{c}_{s}" for c in ("H", "S", "V") for s in ("mean", "std", "skew")]
    + [f"Lesion_{c}_{s}" for c in ("H", "S", "V") for s in ("mean", "std", "skew")]
    + [f"LeafGLCM_{p}" for p in GLCM_PROPS]
    + [f"LesionGLCM_{p}" for p in GLCM_PROPS]
    + ["leaf_ratio", "lesion_ratio", "edge_density"]
)
print(f"{len(FEATURE_NAMES)} features:", FEATURE_NAMES)


def color_moments(hsv_img, mask):
    feats = []
    m = mask > 0
    if m.sum() < 20:
        m = np.ones_like(mask, dtype=bool)
    for ch in range(3):  # H, S, V
        chan = hsv_img[:, :, ch][m].astype(np.float64)
        mean = chan.mean()
        std = chan.std()
        third_moment = ((chan - mean) ** 3).mean()
        skew = np.sign(third_moment) * (abs(third_moment) ** (1 / 3))
        feats.extend([mean, std, skew])
    return feats


def glcm_features(bgr_img, mask):
    gray = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2GRAY)
    m = mask > 0
    if m.sum() < 20:
        m = np.ones_like(mask, dtype=bool)
    roi = np.where(m, gray, 0)
    glcm = graycomatrix(roi, distances=[1], angles=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4],
                         levels=256, symmetric=True, normed=True)
    return [graycoprops(glcm, p).mean() for p in GLCM_PROPS]


def shape_features(leaf_mask, lesion_mask, gray_img):
    total = leaf_mask.size
    leaf_ratio = float((leaf_mask > 0).sum()) / total
    leaf_px = max((leaf_mask > 0).sum(), 1)
    lesion_ratio = float((lesion_mask > 0).sum()) / leaf_px
    edges = cv2.Canny(gray_img, 50, 150)
    edge_density = float((edges > 0).sum()) / total
    return [leaf_ratio, lesion_ratio, edge_density]


def extract_features(path_or_bgr):
    if isinstance(path_or_bgr, str):
        img_bgr = cv2.imread(path_or_bgr)
        if img_bgr is None:
            return None
    else:
        img_bgr = path_or_bgr
    img, hsv, leaf_mask, lesion_mask = segment_leaf_and_lesion(img_bgr)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    feats = []
    feats += color_moments(hsv, leaf_mask)
    feats += color_moments(hsv, lesion_mask)
    feats += glcm_features(img, leaf_mask)
    feats += glcm_features(img, lesion_mask)
    feats += shape_features(leaf_mask, lesion_mask, gray)
    return feats

## 6. Extract features for every image
Cached to Drive as `features.csv` so re-running the notebook later doesn't redo ~5 minutes of extraction. Delete that file from Drive if you ever change the dataset or the feature functions above.

In [ ]:
from tqdm.notebook import tqdm
from joblib import Parallel, delayed

FEATURES_CSV = f"{DRIVE_DIR}/features.csv"

if os.path.exists(FEATURES_CSV):
    print(f"Loading cached features from Drive: {FEATURES_CSV}")
    df = pd.read_csv(FEATURES_CSV)
else:
    def work(row):
        feats = extract_features(row["path"])
        if feats is None:
            return None
        return [row["path"], row["folder"], row["is_leaf"]] + [row[c] for c in ALL_LABELS] + feats

    rows = Parallel(n_jobs=-1)(
        delayed(work)(row) for _, row in tqdm(index_df.iterrows(), total=len(index_df))
    )
    rows = [r for r in rows if r is not None]
    df = pd.DataFrame(rows, columns=["path", "folder", "is_leaf"] + ALL_LABELS + FEATURE_NAMES)
    df.to_csv(FEATURES_CSV, index=False)
    print(f"Saved features to Drive: {FEATURES_CSV}")

print(df.shape)
df.groupby("folder").size()

## 7. Train / validation / test split
70% train, 15% validation (for model selection + threshold tuning), 15% test (final, untouched report). Stratified by the exact folder so every combo and every negative subtype stays balanced across the three splits.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

strat_key = df["folder"].values
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=strat_key, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["folder"].values, random_state=42)
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

X_train = train_df[FEATURE_NAMES].values
X_val = val_df[FEATURE_NAMES].values
X_test = test_df[FEATURE_NAMES].values

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

## 8. Train the classifiers

Two-stage design:
1. **IsLeaf gate** — binary SVM/RF: "is this actually a banana leaf?" Trained on all images
   (positives = every SINGLE_LABEL + MULTI_LABEL image, negatives = every NEGATIVE image).
2. **Per-disease binary classifiers** (Binary Relevance) — one independent SVM/RF per disease,
   trained **only on real leaves**. A leaf can trigger more than one, which is exactly the
   multi-label behavior the panel asked for. If none of the three fire, the leaf is reported Healthy.

For each target, SVM and Random Forest are grid-searched with 5-fold CV and the better one (by
validation F1) is kept — same model-selection idea as the original notebook, just repeated per label.

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score

def grid_search_best(Xtr, ytr, Xv, yv, name):
    svm_grid = GridSearchCV(
        SVC(probability=True, class_weight="balanced"),
        {"C": [1, 10, 50, 100], "gamma": ["scale", 0.01, 0.05, 0.1], "kernel": ["rbf"]},
        cv=5, scoring="f1", n_jobs=-1)
    svm_grid.fit(Xtr, ytr)

    rf_grid = GridSearchCV(
        RandomForestClassifier(random_state=42, class_weight="balanced"),
        {"n_estimators": [200, 300, 400], "max_depth": [None, 15, 25], "min_samples_split": [2, 5]},
        cv=5, scoring="f1", n_jobs=-1)
    rf_grid.fit(Xtr, ytr)

    svm_f1 = f1_score(yv, svm_grid.best_estimator_.predict(Xv))
    rf_f1 = f1_score(yv, rf_grid.best_estimator_.predict(Xv))
    print(f"[{name}] SVM val-F1={svm_f1:.4f} (params={svm_grid.best_params_})")
    print(f"[{name}] RF  val-F1={rf_f1:.4f} (params={rf_grid.best_params_})")

    if svm_f1 >= rf_f1:
        print(f"[{name}] -> chose SVM\n")
        return svm_grid.best_estimator_, "SVM"
    print(f"[{name}] -> chose RandomForest\n")
    return rf_grid.best_estimator_, "RandomForest"


DISEASE_LABELS = ["Cordana", "Sigatoka", "Pestalotiopsis"]

# Stage 1: IsLeaf gate (uses every image)
leaf_model, leaf_model_name = grid_search_best(
    X_train_s, train_df["is_leaf"].values, X_val_s, val_df["is_leaf"].values, "IsLeaf")

# Stage 2: per-disease binary classifiers (leaves only)
leaf_mask_train = train_df["is_leaf"].values == 1
leaf_mask_val = val_df["is_leaf"].values == 1

disease_models, disease_model_names = {}, {}
for d in DISEASE_LABELS:
    model, mname = grid_search_best(
        X_train_s[leaf_mask_train], train_df.loc[leaf_mask_train, d].values,
        X_val_s[leaf_mask_val], val_df.loc[leaf_mask_val, d].values, d)
    disease_models[d] = model
    disease_model_names[d] = mname

## 9. Threshold tuning
Default 0.5 rarely gives the best precision/recall balance. For each classifier, scan thresholds on the **validation set** and keep the one that maximizes F1.

In [ ]:
def best_threshold(y_true, proba):
    best_t, best_f1 = 0.5, -1
    for t in np.arange(0.15, 0.86, 0.01):
        pred = (proba >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return round(float(best_t), 2), best_f1

thresholds = {}

leaf_proba_val = leaf_model.predict_proba(X_val_s)[:, 1]
t, f1v = best_threshold(val_df["is_leaf"].values, leaf_proba_val)
thresholds["IsLeaf"] = t
print(f"IsLeaf best threshold={t}  val-F1={f1v:.4f}")

for d in DISEASE_LABELS:
    yv = val_df.loc[leaf_mask_val, d].values
    proba = disease_models[d].predict_proba(X_val_s[leaf_mask_val])[:, 1]
    t, f1v = best_threshold(yv, proba)
    thresholds[d] = t
    print(f"{d} best threshold={t}  val-F1={f1v:.4f}")

## 10. Final evaluation on the held-out test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support, hamming_loss

y_test_leaf = test_df["is_leaf"].values
leaf_proba_test = leaf_model.predict_proba(X_test_s)[:, 1]
leaf_pred_test = (leaf_proba_test >= thresholds["IsLeaf"]).astype(int)

print("===== IsLeaf gate =====")
print(classification_report(y_test_leaf, leaf_pred_test, target_names=["Negative", "IsLeaf"], digits=3))

leaf_mask_test = test_df["is_leaf"].values == 1
for d in DISEASE_LABELS:
    yv = test_df.loc[leaf_mask_test, d].values
    proba = disease_models[d].predict_proba(X_test_s[leaf_mask_test])[:, 1]
    pred = (proba >= thresholds[d]).astype(int)
    print(f"\n===== {d} (evaluated on real leaves only) =====")
    print(classification_report(yv, pred, target_names=[f"No-{d}", d], digits=3))

In [ ]:
# Full pipeline multi-label metrics: gate + all 3 disease heads combined, exactly how the app will run it
Y_true = test_df[["Healthy"] + DISEASE_LABELS].values
disease_proba_full = {d: disease_models[d].predict_proba(X_test_s)[:, 1] for d in DISEASE_LABELS}

Y_pred = np.zeros_like(Y_true)
for i in range(len(test_df)):
    if leaf_pred_test[i] == 0:
        continue  # not a banana leaf -> everything stays 0
    fired = False
    for j, d in enumerate(DISEASE_LABELS):
        if disease_proba_full[d][i] >= thresholds[d]:
            Y_pred[i, j + 1] = 1
            fired = True
    if not fired:
        Y_pred[i, 0] = 1  # no disease fired -> Healthy

print("Exact-match subset accuracy:", round(float(np.mean(np.all(Y_true == Y_pred, axis=1))), 4))
print("Hamming loss:", round(float(hamming_loss(Y_true, Y_pred)), 4))
print()
for j, c in enumerate(["Healthy"] + DISEASE_LABELS):
    p, r, f1, _ = precision_recall_fscore_support(Y_true[:, j], Y_pred[:, j], average="binary", zero_division=0)
    print(f"  {c:16s} precision={p:.3f} recall={r:.3f} f1={f1:.3f}")

neg_mask_test = test_df["is_leaf"].values == 0
if neg_mask_test.sum() > 0:
    rejected = (leaf_pred_test[neg_mask_test] == 0).mean()
    print(f"\nNegative-image rejection rate: {rejected:.4f}  (n={neg_mask_test.sum()})")

## 11. Visualize results

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
cm = confusion_matrix(y_test_leaf, leaf_pred_test)
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=["Negative", "IsLeaf"],
            yticklabels=["Negative", "IsLeaf"], ax=axes[0])
axes[0].set_title("IsLeaf gate"); axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")

for ax, d in zip(axes[1:], DISEASE_LABELS):
    yv = test_df.loc[leaf_mask_test, d].values
    proba = disease_models[d].predict_proba(X_test_s[leaf_mask_test])[:, 1]
    pred = (proba >= thresholds[d]).astype(int)
    cm = confusion_matrix(yv, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Oranges", xticklabels=[f"No-{d}", d],
                yticklabels=[f"No-{d}", d], ax=ax)
    ax.set_title(d); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

## 12. Save models, scaler, thresholds
Saved both as `.pkl` (Python re-use / grading) and as plain JSON where possible for transparency.

In [ ]:
import joblib

joblib.dump(scaler, "/content/scaler.pkl")
joblib.dump(leaf_model, "/content/is_leaf_model.pkl")
for d in DISEASE_LABELS:
    joblib.dump(disease_models[d], f"/content/{d.lower()}_model.pkl")

with open("/content/thresholds.json", "w") as f:
    json.dump(thresholds, f, indent=2)

with open("/content/feature_names.json", "w") as f:
    json.dump(FEATURE_NAMES, f, indent=2)

with open("/content/model_summary.txt", "w") as f:
    f.write("Banana Leaf Disease Detector — model summary\n")
    f.write("=" * 50 + "\n")
    f.write(f"IsLeaf gate: {leaf_model_name}, threshold={thresholds['IsLeaf']}\n")
    for d in DISEASE_LABELS:
        f.write(f"{d}: {disease_model_names[d]}, threshold={thresholds[d]}\n")
    f.write(f"\nExact-match subset accuracy (test): {float(np.mean(np.all(Y_true == Y_pred, axis=1))):.4f}\n")
    f.write(f"Hamming loss (test): {float(hamming_loss(Y_true, Y_pred)):.4f}\n")

print("Saved .pkl / .json files to /content/")

## 13. Export to ONNX (for the Android app)

`.pkl` files only run inside Python, so they can't be dropped into an Android app. Each sklearn model
is converted to **ONNX**, which the Android app runs with **ONNX Runtime Mobile**. The `StandardScaler`
is just `(x - mean) / scale`, so it's exported as plain JSON numbers instead — trivial to reproduce in
Kotlin without needing a model for it. Every export is verified against the original scikit-learn
predictions before you download anything.

In [ ]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as ort

N_FEATURES = len(FEATURE_NAMES)
initial_type = [("input", FloatTensorType([None, N_FEATURES]))]

with open("/content/scaler_params.json", "w") as f:
    json.dump({"mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist(),
               "feature_names": FEATURE_NAMES}, f, indent=2)

export_models = {"isleaf": leaf_model}
for d in DISEASE_LABELS:
    export_models[d.lower()] = disease_models[d]

X_check = X_val_s[:20].astype(np.float32)
for key, model in export_models.items():
    onnx_model = convert_sklearn(model, initial_types=initial_type,
                                  options={id(model): {"zipmap": False}}, target_opset=15)
    out_path = f"/content/{key}_model.onnx"
    with open(out_path, "wb") as f:
        f.write(onnx_model.SerializeToString())

    sess = ort.InferenceSession(out_path, providers=["CPUExecutionProvider"])
    onnx_proba = sess.run(None, {"input": X_check})[1][:, 1]
    sk_proba = model.predict_proba(X_check)[:, 1]
    max_diff = np.abs(onnx_proba - sk_proba).max()
    print(f"{key}: onnx vs sklearn max prob diff = {max_diff:.6f}")
    assert max_diff < 1e-3, f"ONNX mismatch too large for {key}!"

print("\nAll ONNX exports verified — safe to use in the Android app.")

## 14. Download everything
This downloads the 4 `.onnx` models + `scaler_params.json` + `thresholds.json` + `feature_names.json` that the Android app needs, plus the `.pkl` files and `model_summary.txt` for your records/report.

In [ ]:
from google.colab import files as gfiles

android_files = ["isleaf_model.onnx", "cordana_model.onnx", "sigatoka_model.onnx",
                  "pestalotiopsis_model.onnx", "scaler_params.json", "thresholds.json",
                  "feature_names.json"]
python_files = ["is_leaf_model.pkl", "cordana_model.pkl", "sigatoka_model.pkl",
                 "pestalotiopsis_model.pkl", "scaler.pkl", "model_summary.txt"]

for fname in android_files + python_files:
    gfiles.download(f"/content/{fname}")

shutil.copy(FEATURES_CSV, "/content/features.csv")
gfiles.download("/content/features.csv")

print("Downloaded. The 7 'android_files' above are exactly what the Android Studio project expects")
print("in its assets/ folder — see the app's README_MODEL.md for the drop-in instructions.")

## 15. Quick end-to-end test
Runs the exact same pipeline the app will run: extract features → scale → IsLeaf gate → (if leaf) per-disease scoring.

In [ ]:
def predict_full(path, threshold_dict=thresholds):
    feats = extract_features(path)
    if feats is None:
        return {"error": "could not read image"}
    feats_s = scaler.transform([feats])[0]

    leaf_proba = leaf_model.predict_proba([feats_s])[0][1]
    if leaf_proba < threshold_dict["IsLeaf"]:
        return {"is_leaf": False, "leaf_confidence": round(float(leaf_proba), 3), "diagnosis": "Not a banana leaf"}

    result = {"is_leaf": True, "leaf_confidence": round(float(leaf_proba), 3), "diseases": {}}
    fired = []
    for d in DISEASE_LABELS:
        p = disease_models[d].predict_proba([feats_s])[0][1]
        result["diseases"][d] = round(float(p), 3)
        if p >= threshold_dict[d]:
            fired.append(d)
    result["diagnosis"] = ", ".join(fired) if fired else "Healthy"
    return result


import random
sample_idx = random.sample(range(len(df)), 8)
for i in sample_idx:
    row = df.iloc[i]
    result = predict_full(row["path"])
    print(f'{row["folder"]:32s} -> {result}')